In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import read_KPI_adequacy
dim = (1000,500)


In [ ]:

ss = [
    # {'solution_folder': f"RTS-GMLC_v24.1s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v25.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v28.0s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v28.1s", 'model_type' : 'e-reserve'},
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    ]

gcd_KPI_adequacy, gcdi_KPI_adequacy = read_KPI_adequacy(ss)


In [ ]:
gcd_KPI_adequacy

Data construction and filtering

Following code is to 

In [ ]:
filter_ = gcd_KPI_adequacy.pivot(
    index='day',
    columns = 'model_type',
    values='solution_id',
    # var_name='cost_type',
    # value_name='cost_value'
).dropna().index


In [ ]:
dispatch_data = gcd_KPI_adequacy[gcd_KPI_adequacy['day'].isin(filter_)][['day','model_type', 'storage_charge_uc_MWh', 'storage_discharge_uc_MWh','thermal_production_uc_MWh']]
dispatch_data['storage_charge_discharge_MWh'] = dispatch_data['storage_charge_uc_MWh'] + dispatch_data['storage_discharge_uc_MWh']

In [ ]:

list =['day', 'model_type'] + [col for col in gcdi_KPI_adequacy.columns if '_uc' in col and 'cost' in col]
economic_data = gcd_KPI_adequacy[list]



In [ ]:
def is_approx(x, y, rel_tol=1e-4):
    return abs(x - y) <= rel_tol * max(abs(x), abs(y))

def is_bigger_than(x, y, rel_tol=1e-4):
    return x > y + rel_tol * max(abs(x), abs(y))

economic_data = gcd_KPI_adequacy.pivot(
    index='day',
    columns = 'model_type',
    values=['OV_uc','LGEN_cost_uc','nuclear_production_uc_MWh', 'EOV', 'thermal_production_cost_uc'],
    # var_name='cost_type',
    # value_name='cost_value'
)
economic_data = economic_data.loc[filter_]

# aux = economic_data.pivot(index='day', columns='model_type', values=['OV_uc','LGEN_uc_MWh'])
# # Create a new MultiIndex column for LGEN_cost with same structure as LGEN_uc_MWh
# for col in economic_data['LGEN_uc_MWh'].columns:
# 	economic_data[('LGEN_cost', col)] = economic_data[('LGEN_uc_MWh', col)] * 30
economic_data[('LGEN_%','e-reserve')] = (economic_data[('LGEN_cost_uc','e-reserve')]) / economic_data[('OV_uc','e-reserve')]*100
economic_data[('LGEN_%','envelope')] = (economic_data[('LGEN_cost_uc','envelope')]) / economic_data[('OV_uc','envelope')]*100
economic_data[('thermal_production_cost_uc_%','conservative')] = (economic_data[('thermal_production_cost_uc','conservative')]) / economic_data[('thermal_production_cost_uc','e-reserve')]*100
economic_data[('thermal_production_cost_uc_%','envelope')] = (economic_data[('thermal_production_cost_uc','envelope')]) / economic_data[('thermal_production_cost_uc','e-reserve')]*100

economic_data[('LGEN_%','conservative')] = (economic_data[('LGEN_cost_uc','conservative')] / economic_data[('OV_uc','conservative')])*100
economic_data[('OV_uc_%','conservative')] = (economic_data[('OV_uc','conservative')] - economic_data[('OV_uc','e-reserve')]) / economic_data[('OV_uc','e-reserve')]*100
economic_data[('OV_uc_%','envelope')] = (economic_data[('OV_uc','envelope')] - economic_data[('OV_uc','e-reserve')]) / economic_data[('OV_uc','e-reserve')]*100
economic_data.columns = ['_'.join(col) for col in economic_data.columns]

In [ ]:
economic_data

In [ ]:
thhreshold = 5
def categorize_LGEN(row):
    if (row['LGEN_%_e-reserve'] > thhreshold) and (row['LGEN_%_conservative'] > thhreshold):
        return f"both > {thhreshold}%"
    elif (row['LGEN_%_e-reserve'] <= thhreshold) and (row['LGEN_%_conservative'] > thhreshold):
        return f"only conservative > {thhreshold}%"
    elif (row['LGEN_%_e-reserve'] > thhreshold) and (row['LGEN_%_conservative'] <= thhreshold):
        return f"only e-reserve > {thhreshold}%"
    else:
        return f"both <= {thhreshold}%"
    
# def categorize_nuclear_production(row):
#     if (row['nuclear_production_e-reserve'] > 0) and (row['LGEN_%_conservative'] > thhreshold):
#         return f"both > {thhreshold}%"
#     elif (row['LGEN_%_e-reserve'] <= thhreshold) and (row['LGEN_%_conservative'] > thhreshold):
#         return f"only conservative > {thhreshold}%"
#     elif (row['LGEN_%_e-reserve'] > thhreshold) and (row['LGEN_%_conservative'] <= thhreshold):
#         return f"only e-reserve > {thhreshold}%"
#     else:
#         return f"both <= {thhreshold}%"

economic_data['LGEN_category2'] = economic_data.apply(categorize_LGEN, axis=1)
economic_data['LGEN_category'] = economic_data['LGEN_%_conservative']>thhreshold
economic_data['nuclear_production_category'] = economic_data['nuclear_production_uc_MWh_conservative']/economic_data['nuclear_production_uc_MWh_e-reserve']*100


In [ ]:
# px.scatter(economic_data.reset_index(), y='OV_uc_%_e-reserve', x='day', color = 'LGEN_%_conservative', facet_col = 'LGEN_category2')

In [ ]:
fig = px.scatter(economic_data.reset_index(), y='OV_uc_%_conservative', x='day')
fig.update_layout(title='Cost Difference Conservative w.r.t. E-reserve')
fig.update_layout(
    yaxis_title='cost difference [%]',
    width=dim[0],
    height=dim[1])
fig.show()


In [ ]:

fig = px.scatter(economic_data.reset_index(), y='OV_uc_%_conservative', x='day', 
                color='LGEN_%_conservative', 
                facet_col='LGEN_category',
                color_continuous_scale='Viridis',
                labels={
                    'LGEN_category': 'Loss of Generation Cost > 5%',
                    'LGEN_%_conservative': 'Loss of Generation Cost [%]'
                }
                )
fig.update_layout(
    yaxis_title='cost difference [%]',
    width=dim[0],
    height=dim[1])
fig.show()

In [ ]:

fig = px.scatter(economic_data.reset_index(), y='OV_uc_%_envelope', x='day', 
                color='thermal_production_cost_uc_%_envelope', 
                # facet_col='LGEN_category',
                color_continuous_scale='Viridis',
                labels={
                    'LGEN_category': 'Loss of Generation Cost > 5%',
                    'LGEN_%_envelope': 'Loss of Generation Cost [%]'
                }
                )
fig.update_layout(
    yaxis_title='cost difference [%]',
    width=dim[0],
    height=dim[1])
fig.show()

In [ ]:
fig = px.scatter(economic_data.reset_index(), y='OV_uc_%_conservative', x='day', color = 'nuclear_production_category',
           color_continuous_scale='Viridis',
                labels={
                    'nuclear_production_category': 'Nuclear production ratio [%]',
                    # 'LGEN_%_conservative': 'Loss of Generation Cost [%]'
                })
fig.update_layout(
    yaxis_title='cost difference [%]',
    width=dim[0],
    height=dim[1])
fig.show()

In [ ]:
economic_data

In [ ]:
# Remove the backspace character from stats_text

fig = px.box(economic_data, y='OV_uc_%_conservative')
fig.update_layout(
    yaxis_title='cost difference [%]',
    width=dim[0]/2,
    height=dim[1])
fig.update_traces(
    boxmean=True,
)
stats_text = (f"Mean: {economic_data['OV_uc_%_conservative'].mean():.2f}%\n"
              f"Median: {economic_data['OV_uc_%_conservative'].median():.2f}%\n"
              f"Min: {economic_data['OV_uc_%_conservative'].quantile(0):.2f}%\n"
              f"Max: {economic_data['OV_uc_%_conservative'].quantile(1):.2f}%"
            #   f"25th percentile: {economic_data['OV_uc_%_e-reserve'].quantile(0.25):.2f}%\n"
            #   f"75th percentile: {economic_data['OV_uc_%_e-reserve'].quantile(0.75):.2f}%")
)
fig.add_annotation(
    text=stats_text,
    xref="paper", yref="paper",
    x=1., y=-0.1,
    showarrow=False,
    font=dict(size=12)
)

In [ ]:

fig = px.box(economic_data, y='OV_uc_%_envelope')
fig.update_layout(
    yaxis_title='cost difference [%]',
    width=dim[0]/2,
    height=dim[1])
fig.update_traces(
    boxmean=True,
)
stats_text = (f"Mean: {economic_data['OV_uc_%_envelope'].mean():.2f}%\n"
              f"Median: {economic_data['OV_uc_%_envelope'].median():.2f}%\n"
              f"Min: {economic_data['OV_uc_%_envelope'].quantile(0):.2f}%\n"
              f"Max: {economic_data['OV_uc_%_envelope'].quantile(1):.2f}%"
            #   f"25th percentile: {economic_data['OV_uc_%_e-reserve'].quantile(0.25):.2f}%\n"
            #   f"75th percentile: {economic_data['OV_uc_%_e-reserve'].quantile(0.75):.2f}%")
)
fig.add_annotation(
    text=stats_text,
    xref="paper", yref="paper",
    x=1., y=-0.1,
    showarrow=False,
    font=dict(size=12)
)